# AdaLi vs Sigmoid Comparison (MNIST, CIFAR-10, CIFAR-100)

This notebook compares **AdaLi** and **Sigmoid** surrogate activations from this repository using:
- Datasets: MNIST, CIFAR-10, CIFAR-100
- Architectures: a simple CNN and a small ResNet-style model

In [2]:
import os
import copy
import time
import random
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms

from config import AdaLiConfig
from modules.sgAdaLi import AdaLi
from modules.sgSigmoid import Sigmoid

print('Torch:', torch.__version__)
print('Torchvision:', torchvision.__version__)

Torch: 2.11.0+cpu
Torchvision: 0.26.0+cpu


In [3]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

# Keep a stable copy of the original AdaLi config values.
BASE_ADALI_CFG = copy.deepcopy(AdaLiConfig)
BASE_ADALI_CFG

Device: cpu


{'Epoch': 500, 'epoch': 1, 'Vth': 1.0, 'Left': [1.5, 0.3], 'Right': [1.5, 0.3]}

In [4]:
@dataclass
class ExperimentConfig:
    data_root: str = './data'
    batch_size: int = 128
    num_workers: int = 2
    epochs: int = 3
    lr: float = 1e-3
    weight_decay: float = 1e-4

    # Quick mode: limit dataset size for faster prototyping.
    train_subset: int | None = 20000
    test_subset: int | None = 5000

    adali_alpha: float = 0.5
    adali_beta: float = 0.5
    adali_left: float = 1.5
    adali_right: float = 1.5

    sigmoid_alpha: float = 4.0

cfg = ExperimentConfig()
cfg

ExperimentConfig(data_root='./data', batch_size=128, num_workers=2, epochs=3, lr=0.001, weight_decay=0.0001, train_subset=20000, test_subset=5000, adali_alpha=0.5, adali_beta=0.5, adali_left=1.5, adali_right=1.5, sigmoid_alpha=4.0)

In [ ]:
def _maybe_subset(dataset, n, seed=SEED):
    if n is None or n >= len(dataset):
        return dataset
    g = torch.Generator()
    g.manual_seed(seed)
    idx = torch.randperm(len(dataset), generator=g)[:n].tolist()
    return Subset(dataset, idx)


def _build_dataset(dataset_cls, root, train, transform, retries=3, base_wait_s=3):
    import urllib.error

    last_err = None
    for attempt in range(1, retries + 1):
        try:
            return dataset_cls(root, train=train, download=True, transform=transform)
        except (urllib.error.HTTPError, urllib.error.URLError) as err:
            last_err = err
            if attempt == retries:
                break
            wait_s = base_wait_s * attempt
            print(
                f"Download issue for {dataset_cls.__name__} (train={train}) - "
                f"attempt {attempt}/{retries}: {err}. Retrying in {wait_s}s..."
            )
            time.sleep(wait_s)

    # If files are already present locally, this succeeds without network.
    try:
        print(f"Using local cache for {dataset_cls.__name__} (train={train}).")
        return dataset_cls(root, train=train, download=False, transform=transform)
    except Exception as cache_err:
        raise RuntimeError(
            f"Failed to prepare {dataset_cls.__name__} (train={train}). "
            f"Last download error: {last_err}. Cache error: {cache_err}"
        ) from last_err


def get_dataloaders(dataset_name: str, cfg: ExperimentConfig):
    dataset_name = dataset_name.lower()

    if dataset_name == 'mnist':
        transform_train = transforms.Compose([
            transforms.RandomCrop(28, padding=2),
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
        ])
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,)),
        ])
        train_set = _build_dataset(torchvision.datasets.MNIST, cfg.data_root, train=True, transform=transform_train)
        test_set = _build_dataset(torchvision.datasets.MNIST, cfg.data_root, train=False, transform=transform_test)
        in_channels = 1
        num_classes = 10

    elif dataset_name == 'cifar10':
        mean = (0.4914, 0.4822, 0.4465)
        std = (0.2470, 0.2435, 0.2616)
        transform_train = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        train_set = _build_dataset(torchvision.datasets.CIFAR10, cfg.data_root, train=True, transform=transform_train)
        test_set = _build_dataset(torchvision.datasets.CIFAR10, cfg.data_root, train=False, transform=transform_test)
        in_channels = 3
        num_classes = 10

    elif dataset_name == 'cifar100':
        mean = (0.5071, 0.4867, 0.4408)
        std = (0.2675, 0.2565, 0.2761)
        transform_train = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
        train_set = _build_dataset(torchvision.datasets.CIFAR100, cfg.data_root, train=True, transform=transform_train)
        test_set = _build_dataset(torchvision.datasets.CIFAR100, cfg.data_root, train=False, transform=transform_test)
        in_channels = 3
        num_classes = 100

    else:
        raise ValueError(f'Unsupported dataset: {dataset_name}')

    train_set = _maybe_subset(train_set, cfg.train_subset)
    test_set = _maybe_subset(test_set, cfg.test_subset)

    train_loader = DataLoader(
        train_set,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    test_loader = DataLoader(
        test_set,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=torch.cuda.is_available(),
    )

    return train_loader, test_loader, in_channels, num_classes

In [6]:
def set_surrogate_mode(name: str, cfg: ExperimentConfig):
    # Reset global config first to avoid cross-run side effects.
    AdaLiConfig['Vth'] = BASE_ADALI_CFG['Vth']
    AdaLiConfig['Left'][0] = BASE_ADALI_CFG['Left'][0]
    AdaLiConfig['Right'][0] = BASE_ADALI_CFG['Right'][0]

    if name.lower() == 'adali':
        AdaLiConfig['Left'][0] = cfg.adali_left
        AdaLiConfig['Right'][0] = cfg.adali_right
    elif name.lower() == 'sigmoid':
        # Sigmoid constructor sets Left/Right to inf internally.
        pass
    else:
        raise ValueError(f'Unsupported surrogate: {name}')


def activation_factory(name: str, cfg: ExperimentConfig):
    name = name.lower()

    def _make():
        if name == 'adali':
            return AdaLi(alpha=cfg.adali_alpha, beta=cfg.adali_beta)
        if name == 'sigmoid':
            return Sigmoid(alpha=cfg.sigmoid_alpha)
        raise ValueError(f'Unsupported surrogate: {name}')

    return _make

In [7]:
class SimpleCNN(nn.Module):
    def __init__(self, in_channels: int, num_classes: int, act_factory):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            act_factory(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            act_factory(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            act_factory(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


class BasicBlock(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int, act_factory):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.act1 = act_factory()

        self.conv2 = nn.Conv2d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.act2 = act_factory()

        self.shortcut = nn.Identity()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch),
            )

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.act1(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = out + identity
        out = self.act2(out)
        return out


class SmallResNet(nn.Module):
    def __init__(self, in_channels: int, num_classes: int, act_factory):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            act_factory(),
        )

        self.layer1 = nn.Sequential(
            BasicBlock(32, 32, stride=1, act_factory=act_factory),
            BasicBlock(32, 32, stride=1, act_factory=act_factory),
        )
        self.layer2 = nn.Sequential(
            BasicBlock(32, 64, stride=2, act_factory=act_factory),
            BasicBlock(64, 64, stride=1, act_factory=act_factory),
        )
        self.layer3 = nn.Sequential(
            BasicBlock(64, 128, stride=2, act_factory=act_factory),
            BasicBlock(128, 128, stride=1, act_factory=act_factory),
        )

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


def build_model(model_name: str, in_channels: int, num_classes: int, act_factory):
    name = model_name.lower()
    if name == 'simple_cnn':
        return SimpleCNN(in_channels=in_channels, num_classes=num_classes, act_factory=act_factory)
    if name == 'small_resnet':
        return SmallResNet(in_channels=in_channels, num_classes=num_classes, act_factory=act_factory)
    raise ValueError(f'Unsupported model: {model_name}')

In [8]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        running_loss += loss.item() * labels.size(0)
        preds = logits.argmax(dim=1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, running_correct / total


def run_experiment(dataset_name: str, model_name: str, surrogate_name: str, cfg: ExperimentConfig):
    set_surrogate_mode(surrogate_name, cfg)

    train_loader, test_loader, in_channels, num_classes = get_dataloaders(dataset_name, cfg)
    act_factory = activation_factory(surrogate_name, cfg)

    model = build_model(model_name, in_channels=in_channels, num_classes=num_classes, act_factory=act_factory).to(DEVICE)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    history = []
    start_time = time.time()

    for epoch in range(1, cfg.epochs + 1):
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
        test_loss, test_acc = evaluate(model, test_loader, criterion, DEVICE)

        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'train_acc': train_acc,
            'test_loss': test_loss,
            'test_acc': test_acc,
        })

        print(f'[{dataset_name:8s}] [{model_name:11s}] [{surrogate_name:7s}] Epoch {epoch:2d}/{cfg.epochs} | '
              f'train_acc={train_acc:.4f} test_acc={test_acc:.4f}')

    elapsed = time.time() - start_time
    best_test_acc = max(h['test_acc'] for h in history)

    return {
        'dataset': dataset_name,
        'model': model_name,
        'surrogate': surrogate_name,
        'epochs': cfg.epochs,
        'best_test_acc': best_test_acc,
        'final_test_acc': history[-1]['test_acc'],
        'train_subset': cfg.train_subset,
        'test_subset': cfg.test_subset,
        'seconds': elapsed,
        'history': history,
    }

In [ ]:
# Main benchmark grid
datasets = ['mnist','cifar10', 'cifar100']  
models = ['simple_cnn', 'small_resnet']
surrogates = ['adali', 'sigmoid']

# Keep prior progress if this cell is re-run.
if 'all_runs' not in globals() or not isinstance(all_runs, list):
    all_runs = []

done = {(r['dataset'], r['model'], r['surrogate']) for r in all_runs}
failed_runs = []

# Preflight each dataset once to avoid repeated retries per model/surrogate pair.
ready_datasets = []
for ds in datasets:
    try:
        _ = get_dataloaders(ds, cfg)
        ready_datasets.append(ds)
        print(f'Dataset ready: {ds}')
    except Exception as err:
        print(f'Skipping dataset {ds} for now (data unavailable): {err}')

for ds in ready_datasets:
    for model_name in models:
        for surr in surrogates:
            key = (ds, model_name, surr)
            if key in done:
                print(f'Skipping already completed: {key}')
                continue

            try:
                result = run_experiment(ds, model_name, surr, cfg)
                all_runs.append(result)
                done.add(key)
            except Exception as err:
                msg = str(err)
                failed_runs.append({'dataset': ds, 'model': model_name, 'surrogate': surr, 'error': msg})
                print(f'Failed {key}: {msg}')

print(f'Total completed runs: {len(all_runs)}')
if failed_runs:
    print(f'Failed runs in this pass: {len(failed_runs)}')
    for fr in failed_runs:
        print(f"- ({fr['dataset']}, {fr['model']}, {fr['surrogate']})")

Download issue for CIFAR10 (train=True) - attempt 1/3: HTTP Error 503: Service Unavailable. Retrying in 3s...
Download issue for CIFAR10 (train=True) - attempt 2/3: HTTP Error 503: Service Unavailable. Retrying in 6s...
Using local cache for CIFAR10 (train=True).
Failed ('cifar10', 'simple_cnn', 'adali'): Failed to prepare CIFAR10 (train=True). Last download error: HTTP Error 503: Service Unavailable. Cache error: Dataset not found or corrupted. You can use download=True to download it
Download issue for CIFAR10 (train=True) - attempt 1/3: HTTP Error 503: Service Unavailable. Retrying in 3s...
Download issue for CIFAR10 (train=True) - attempt 2/3: HTTP Error 503: Service Unavailable. Retrying in 6s...
Using local cache for CIFAR10 (train=True).
Failed ('cifar10', 'simple_cnn', 'sigmoid'): Failed to prepare CIFAR10 (train=True). Last download error: HTTP Error 503: Service Unavailable. Cache error: Dataset not found or corrupted. You can use download=True to download it
Download issue f

KeyboardInterrupt: 

In [ ]:
records = []
for r in all_runs:
    records.append({
        'dataset': r['dataset'],
        'model': r['model'],
        'surrogate': r['surrogate'],
        'best_test_acc': r['best_test_acc'],
        'final_test_acc': r['final_test_acc'],
        'seconds': r['seconds'],
        'epochs': r['epochs'],
        'train_subset': r['train_subset'],
        'test_subset': r['test_subset'],
    })

df = pd.DataFrame(records).sort_values(['dataset', 'model', 'surrogate']).reset_index(drop=True)
df

In [ ]:
pivot = df.pivot_table(
    index=['dataset', 'model'],
    columns='surrogate',
    values='best_test_acc',
    aggfunc='max'
).reset_index()

if {'adali', 'sigmoid'}.issubset(set(pivot.columns)):
    pivot['adali_minus_sigmoid'] = pivot['adali'] - pivot['sigmoid']

pivot

In [ ]:
# Visual comparison
plot_df = df.copy()
plot_df['label'] = plot_df['dataset'] + ' | ' + plot_df['model']

plt.figure(figsize=(12, 5))
for surrogate_name, grp in plot_df.groupby('surrogate'):
    x = np.arange(len(grp))

# Keep deterministic order across groups
plot_df = plot_df.sort_values(['dataset', 'model', 'surrogate']).reset_index(drop=True)
labels = sorted(plot_df['label'].unique())
x = np.arange(len(labels))
width = 0.35

vals_adali = []
vals_sigmoid = []
for lbl in labels:
    vals_adali.append(float(plot_df[(plot_df['label'] == lbl) & (plot_df['surrogate'] == 'adali')]['best_test_acc'].iloc[0]))
    vals_sigmoid.append(float(plot_df[(plot_df['label'] == lbl) & (plot_df['surrogate'] == 'sigmoid')]['best_test_acc'].iloc[0]))

plt.bar(x - width / 2, vals_adali, width=width, label='AdaLi')
plt.bar(x + width / 2, vals_sigmoid, width=width, label='Sigmoid')
plt.xticks(x, labels, rotation=30, ha='right')
plt.ylabel('Best Test Accuracy')
plt.title('AdaLi vs Sigmoid across Dataset/Architecture')
plt.ylim(0, 1.0)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Save result table for reporting
out_path = 'adali_vs_sigmoid_results.csv'
df.to_csv(out_path, index=False)
print('Saved:', out_path)